In [1]:
import pygame
import random
import math

import sys

pygame 2.6.1 (SDL 2.28.4, Python 3.12.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [5]:
pygame.init()

WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Track Racer")

track_img = pygame.image.load("race_track_withfinishandcheckpoint.png")
track_img = pygame.transform.scale(track_img, (WIDTH, HEIGHT))

car_img = pygame.image.load("Car_Blue.png")
car_img = pygame.transform.scale(car_img, (50, 30))

# Find a pixel on the start/finish line to place the car
start_x, start_y = 400, 300  # Adjust based on your image
car_x = start_x
car_y = start_y
car_angle = 0
car_speed = 0
max_speed = 5
acceleration = 0.1
deceleration = 0.2
turn_speed = 3

clock = pygame.time.Clock()
FPS = 60

lap_count = 0
checkpoint_count = 0
lap_triggered = False
checkpoint_triggered = [False, False, False]  # For 3 checkpoints

def rotate_car(image, angle):
    return pygame.transform.rotate(image, angle)

def get_car_center(x, y, angle):
    rad = math.radians(angle)
    offset_x = math.cos(rad) * 25
    offset_y = -math.sin(rad) * 15
    return int(x + offset_x), int(y + offset_y)

def is_on_track(x, y):
    if 0 <= x < WIDTH and 0 <= y < HEIGHT:
        r, g, b = track_img.get_at((x, y))[:3]
        return (r in range(40, 80)) and (g in range(40, 80)) and (b in range(40, 80))
    return False

def is_finish_line(x, y):
    r, g, b = track_img.get_at((x, y))[:3]
    return r > 250 and g > 240 and b < 100  # Yellow

def is_checkpoint(x, y):
    r, g, b = track_img.get_at((x, y))[:3]
    return r in range(60, 70) and g in range(70, 80) and b > 200  # Blue-purple

# Game loop
running = True
while running:
    screen.blit(track_img, (0, 0))

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    keys = pygame.key.get_pressed()
    if keys[pygame.K_w]:
        car_speed = min(car_speed + acceleration, max_speed)
    elif keys[pygame.K_s]:
        car_speed = max(car_speed - deceleration, -max_speed / 2)
    else:
        car_speed *= 0.98

    if keys[pygame.K_a]:
        car_angle += turn_speed
    if keys[pygame.K_d]:
        car_angle -= turn_speed

    rad = math.radians(car_angle)
    car_x += math.cos(rad) * car_speed
    car_y -= math.sin(rad) * car_speed

    car_x = max(0, min(WIDTH, car_x))
    car_y = max(0, min(HEIGHT, car_y))

    center_x, center_y = get_car_center(car_x, car_y, car_angle)
    on_track = is_on_track(center_x, center_y)

    if not on_track:
        car_speed *= 0.25

    # Checkpoint logic
    if is_checkpoint(center_x, center_y):
        for i in range(3):
            if not checkpoint_triggered[i]:
                checkpoint_triggered[i] = True
                checkpoint_count += 1
                print(f"Checkpoint {i+1} reached!")
                break

    # Lap logic
    if is_finish_line(center_x, center_y):
        if not lap_triggered and checkpoint_count >= 3:
            lap_count += 1
            print(f"Lap {lap_count} complete!")
            checkpoint_count = 0
            checkpoint_triggered = [False, False, False]
            lap_triggered = True
    else:
        lap_triggered = False

    rotated_car = rotate_car(car_img, car_angle)
    rect = rotated_car.get_rect(center=(car_x, car_y))
    screen.blit(rotated_car, rect.topleft)

    pygame.draw.circle(screen, (255, 0, 0), (center_x, center_y), 3)
    if not on_track:
        pygame.draw.rect(screen, (255, 0, 0), rect, 2)

    pygame.display.update()
    clock.tick(FPS)

pygame.quit()

Checkpoint 1 reached!
Checkpoint 2 reached!
Checkpoint 3 reached!
Lap 1 complete!
Checkpoint 1 reached!
Checkpoint 2 reached!
Checkpoint 3 reached!


In [7]:
# Initialize Pygame
pygame.init()

# Screen setup
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Track Racer")

track_img = pygame.image.load("race_track_withfinishandcheckpoint.png")
track_img = pygame.transform.scale(track_img, (WIDTH, HEIGHT))

car_img = pygame.image.load("Car_Blue.png")
car_img = pygame.transform.scale(car_img, (50, 30))

# Car starting position (adjust to match yellow line on your image)
car_x = 400
car_y = 300
car_angle = 0
car_speed = 0
max_speed = 5
acceleration = 0.1
deceleration = 0.2
turn_speed = 3

# Lap and checkpoint tracking
lap_count = 0
checkpoint_count = 0
lap_triggered = False
checkpoint_triggered = [False, False, False]

# Define checkpoint zones (adjust these to match your image)
checkpoint_zones = [
    pygame.Rect(150, 200, 40, 40),  # Checkpoint 1
    pygame.Rect(400, 100, 40, 40),  # Checkpoint 2
    pygame.Rect(600, 300, 40, 40),  # Checkpoint 3
]

# Clock
clock = pygame.time.Clock()
FPS = 60

# Helper functions
def rotate_car(image, angle):
    return pygame.transform.rotate(image, angle)

def get_car_center(x, y, angle):
    rad = math.radians(angle)
    offset_x = math.cos(rad) * 25
    offset_y = -math.sin(rad) * 15
    return int(x + offset_x), int(y + offset_y)

def is_on_track(x, y):
    if 0 <= x < WIDTH and 0 <= y < HEIGHT:
        r, g, b = track_img.get_at((x, y))[:3]
        return (r in range(40, 80)) and (g in range(40, 80)) and (b in range(40, 80))
    return False

def is_finish_line(x, y):
    r, g, b = track_img.get_at((x, y))[:3]
    return r > 250 and g > 240 and b < 100  # Yellow

def check_checkpoint(x, y):
    for i, zone in enumerate(checkpoint_zones):
        if zone.collidepoint(x, y) and not checkpoint_triggered[i]:
            checkpoint_triggered[i] = True
            print(f"Checkpoint {i+1} reached!")
            return True
    return False

# Game loop
running = True
while running:
    screen.blit(track_img, (0, 0))

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Controls
    keys = pygame.key.get_pressed()
    if keys[pygame.K_w]:
        car_speed = min(car_speed + acceleration, max_speed)
    elif keys[pygame.K_s]:
        car_speed = max(car_speed - deceleration, -max_speed / 2)
    else:
        car_speed *= 0.98

    if keys[pygame.K_a]:
        car_angle += turn_speed
    if keys[pygame.K_d]:
        car_angle -= turn_speed

    # Movement
    rad = math.radians(car_angle)
    car_x += math.cos(rad) * car_speed
    car_y -= math.sin(rad) * car_speed

    # Clamp to screen
    car_x = max(0, min(WIDTH, car_x))
    car_y = max(0, min(HEIGHT, car_y))

    # Collision check
    center_x, center_y = get_car_center(car_x, car_y, car_angle)
    on_track = is_on_track(center_x, center_y)

    if not on_track:
        car_speed *= 0.25

    # Checkpoint logic
    if check_checkpoint(center_x, center_y):
        checkpoint_count += 1

    # Lap logic
    if is_finish_line(center_x, center_y):
        if not lap_triggered and checkpoint_count >= 3:
            lap_count += 1
            print(f"Lap {lap_count} complete!")
            checkpoint_count = 0
            checkpoint_triggered = [False, False, False]
            lap_triggered = True
    else:
        lap_triggered = False

    # Draw car
    rotated_car = rotate_car(car_img, car_angle)
    rect = rotated_car.get_rect(center=(car_x, car_y))
    screen.blit(rotated_car, rect.topleft)

    # Visual debug
    pygame.draw.circle(screen, (255, 0, 0), (center_x, center_y), 3)
    if not on_track:
        pygame.draw.rect(screen, (255, 0, 0), rect, 2)

    # Draw checkpoint zones
    for zone in checkpoint_zones:
        pygame.draw.rect(screen, (63, 72, 204), zone, 2)

    pygame.display.update()
    clock.tick(FPS)

pygame.quit()

Checkpoint 3 reached!
Checkpoint 2 reached!
Checkpoint 1 reached!
Lap 1 complete!
Checkpoint 3 reached!
